<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/contact_extractor_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os

print("Project files:")
print(os.listdir())

print("\nData files:")
print(os.listdir("data"))

Project files:
['data', '__pycache__', 'osm_scraper.py', 'online_presence_checker.py', 'website_analyzer.py', 'config.py', 'scorer.py']

Data files:
['healthcare_osm_raw.csv', 'healthcare_online_presence.csv', 'healthcare_website_analysis.csv', 'healthcare_scored_leads.csv']


In [ ]:
import os
import pandas as pd
from config import SCORED_LEADS_CSV

print("Step 5 input file:")
print(SCORED_LEADS_CSV)

print("\nDoes Step 5 input file exist?")
print(os.path.exists(SCORED_LEADS_CSV))

df4 = pd.read_csv(SCORED_LEADS_CSV)
print("\nRows:", len(df4))
print("\nColumns:")
print(df4.columns.tolist())

Step 5 input file:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv

Does Step 5 input file exist?
True

Rows: 100

Columns:
['Unnamed: 0', 'Business Name', 'Industry Category', 'Business Description', 'Location', 'Latitude', 'Longitude', 'Google Maps Profile Link', 'OpenStreetMap Link', 'Website URL', 'Phone Number', 'Email Address', 'Source', 'Official Website Found', 'Online Presence Type', 'Referral Platform Links', 'Search Result Links', 'Website Classification', 'Lead Score', 'Business Potential Category', 'Detailed Score Reason', 'Reason for Classification']


In [ ]:
%%writefile config.py

import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")
WEBSITE_ANALYSIS_CSV = os.path.join(DATA_DIR, "healthcare_website_analysis.csv")
SCORED_LEADS_CSV = os.path.join(DATA_DIR, "healthcare_scored_leads.csv")

# Step 5 output
CONTACT_ENRICHED_CSV = os.path.join(DATA_DIR, "healthcare_contact_enriched.csv")

# Final output
FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [ ]:
import importlib
import config
importlib.reload(config)
from config import SCORED_LEADS_CSV, CONTACT_ENRICHED_CSV
import os

print("Step 5 input:", SCORED_LEADS_CSV)
print("Input exists:", os.path.exists(SCORED_LEADS_CSV))

print("\nStep 5 output will be:", CONTACT_ENRICHED_CSV)

Step 5 input: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv
Input exists: True

Step 5 output will be: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_contact_enriched.csv


In [ ]:
!pip install requests beautifulsoup4 pandas -q

In [ ]:
%%writefile contact_extractor.py

import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from config import SCORED_LEADS_CSV, CONTACT_ENRICHED_CSV


HEADERS = {
    "User-Agent": "Mozilla/5.0 HealthcareLeadGenBot/1.0"
}


def clean_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_url(url):
    url = clean_value(url)

    if not url:
        return ""

    if not url.startswith("http://") and not url.startswith("https://"):
        url = "https://" + url

    return url


def fetch_html(url):
    url = normalize_url(url)

    if not url:
        return "", ""

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10,
            allow_redirects=True
        )

        if response.status_code < 400:
            return response.text, response.url

    except Exception:
        pass

    return "", ""


def extract_emails(text):
    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    emails = re.findall(email_pattern, text)

    clean_emails = []
    for email in emails:
        email = email.strip().lower()
        if email not in clean_emails:
            clean_emails.append(email)

    return clean_emails


def extract_indian_phone_numbers(text):
    """
    Extracts Indian phone numbers from public website text.
    """
    phone_patterns = [
        r"\+91[\-\s]?[6-9]\d{9}",
        r"\+91[\-\s]?[6-9]\d{4}[\-\s]?\d{5}",
        r"\b[6-9]\d{9}\b",
        r"\b0\d{2,5}[\-\s]?\d{6,8}\b"
    ]

    phones = []

    for pattern in phone_patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            if isinstance(match, tuple):
                match = "".join(match)

            phone = str(match).strip()
            phone = re.sub(r"\s+", " ", phone)

            if phone not in phones:
                phones.append(phone)

    return phones


def extract_linkedin_links(soup, base_url):
    links = []

    if soup is None:
        return links

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        if "linkedin.com" in href.lower():
            full_link = urljoin(base_url, href)

            if full_link not in links:
                links.append(full_link)

    return links


def extract_owner_founder_name(text):
    """
    Basic public-text pattern search.
    If not clearly found, returns blank.
    """
    text = re.sub(r"\s+", " ", text)

    patterns = [
        r"(Founder|Owner|Director|Managing Director|Chairman)\s*[:\-]\s*([A-Z][A-Za-z.\s]{2,60})",
        r"([A-Z][A-Za-z.\s]{2,60})\s*,?\s*(Founder|Owner|Director|Managing Director|Chairman)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text)

        if match:
            groups = match.groups()

            if len(groups) >= 2:
                if groups[0] in ["Founder", "Owner", "Director", "Managing Director", "Chairman"]:
                    name = groups[1]
                else:
                    name = groups[0]

                name = name.strip()

                # Avoid very long bad matches
                if 2 <= len(name.split()) <= 6:
                    return name

    return ""


def extract_contacts_from_website(website_url):
    website_url = normalize_url(website_url)

    if not website_url:
        return {
            "website_phone": "",
            "website_email": "",
            "linkedin": "",
            "owner_founder": "",
            "status": "No website available for contact extraction"
        }

    html, final_url = fetch_html(website_url)

    if not html:
        return {
            "website_phone": "",
            "website_email": "",
            "linkedin": "",
            "owner_founder": "",
            "status": "Website could not be opened for contact extraction"
        }

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)

    emails = extract_emails(text)
    phones = extract_indian_phone_numbers(text)
    linkedin_links = extract_linkedin_links(soup, final_url)
    owner_founder = extract_owner_founder_name(text)

    return {
        "website_phone": phones[0] if phones else "",
        "website_email": emails[0] if emails else "",
        "linkedin": linkedin_links[0] if linkedin_links else "",
        "owner_founder": owner_founder,
        "status": "Contact extraction completed"
    }


def enrich_contacts(input_file=SCORED_LEADS_CSV, output_file=CONTACT_ENRICHED_CSV):
    print("Step 5 started")
    print("Input file:", input_file)
    print("Output file:", output_file)

    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input file not found: {input_file}. Please complete Step 4 first."
        )

    df = pd.read_csv(input_file)

    final_phones = []
    final_emails = []
    linkedin_profiles = []
    owner_founder_names = []
    extraction_statuses = []

    print("Total records to enrich:", len(df))

    for index, row in df.iterrows():
        business_name = clean_value(row.get("Business Name", ""))

        print(f"[{index + 1}/{len(df)}] Extracting contact for: {business_name}")

        existing_phone = clean_value(row.get("Phone Number", ""))
        existing_email = clean_value(row.get("Email Address", ""))

        official_website = clean_value(row.get("Official Website Found", ""))

        if not official_website:
            official_website = clean_value(row.get("Website URL", ""))

        extracted = extract_contacts_from_website(official_website)

        final_phone = existing_phone if existing_phone else extracted["website_phone"]
        final_email = existing_email if existing_email else extracted["website_email"]

        final_phones.append(final_phone)
        final_emails.append(final_email)
        linkedin_profiles.append(extracted["linkedin"])
        owner_founder_names.append(extracted["owner_founder"])
        extraction_statuses.append(extracted["status"])

        time.sleep(1)

    df["Final Phone Number"] = final_phones
    df["Final Email Address"] = final_emails
    df["LinkedIn Profile"] = linkedin_profiles
    df["Owner / Founder Name"] = owner_founder_names
    df["Contact Extraction Status"] = extraction_statuses

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Step 5 completed")
    print("Saved file:", output_file)

    return df


if __name__ == "__main__":
    df = enrich_contacts(
        input_file=SCORED_LEADS_CSV,
        output_file=CONTACT_ENRICHED_CSV
    )

    columns_to_show = [
        "Business Name",
        "Final Phone Number",
        "Final Email Address",
        "LinkedIn Profile",
        "Owner / Founder Name",
        "Contact Extraction Status"
    ]

    available_columns = [col for col in columns_to_show if col in df.columns]

    print("\nPreview:")
    print(df[available_columns].head())

Writing contact_extractor.py


In [ ]:
!python contact_extractor.py

Step 5 started
Input file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv
Output file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_contact_enriched.csv
Total records to enrich: 100
[1/100] Extracting contact for: Greater Kailash Hospital
[2/100] Extracting contact for: Qurban Husain
[3/100] Extracting contact for: Shri Aurobindo dental clinic and institute of medical science
[4/100] Extracting contact for: Dr. Mutha
[5/100] Extracting contact for: Arshid shah
[6/100] Extracting contact for: Shubham Hospital
[7/100] Extracting contact for: Aditya Nursing Home
[8/100] Extracting contact for: Curewell hospital janjeerwala square
[9/100] Extracting contact for: kjd
[10/100] Extracting contact for: Family Dental Clinic
[11/100] Extracting contact for: Yoga Amrutam
[12/100] Extracting contact for: Suhan Pathology
[13/100] Extracting contact for: Dr. Lal Path Labs
[14/100] Extracting contact for: Dr. Vrinda Vashistha
[15/100] Extract

In [ ]:
import os
import pandas as pd
from config import CONTACT_ENRICHED_CSV

print("Step 5 file exists:", os.path.exists(CONTACT_ENRICHED_CSV))
print("Step 5 file path:", CONTACT_ENRICHED_CSV)

df5 = pd.read_csv(CONTACT_ENRICHED_CSV)

print("Total rows:", len(df5))

df5[[
    "Business Name",
    "Final Phone Number",
    "Final Email Address",
    "LinkedIn Profile",
    "Owner / Founder Name",
    "Contact Extraction Status"
]].head()

Step 5 file exists: True
Step 5 file path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_contact_enriched.csv
Total rows: 100


,Business Name,Final Phone Number,Final Email Address,LinkedIn Profile,Owner / Founder Name,Contact Extraction Status
0,Greater Kailash Hospital,NaN,NaN,NaN,NaN,Website could not be opened for contact extrac...
1,Qurban Husain,NaN,crd@tih.org.pk,https://www.linkedin.com/company/the-indus-hos...,Blood Transfusion Services Directorate,Contact extraction completed
2,Shri Aurobindo dental clinic and institute of ...,6232007712,NaN,NaN,NaN,Contact extraction completed
3,Dr. Mutha,+9198-26-150111,NaN,NaN,Gita Bhawan Trust Gopaldas Mittal,Contact extraction completed
4,Arshid shah,8103012446,shah.arshid05@rediffmail.com,NaN,NaN,Website could not be opened for contact extrac...
